# Member B 实验记录：Baseline 与模型改进

**任务**：用蛋白质/肽数据预测帕金森患者未来 0/6/12/24 个月的 MDS-UPDRS 评分（updrs_1~4）。
**协议**：组里冻结的 5 折交叉验证（患者不跨折），指标 SMAPE+1（越小越好），12,697 条预测。
**配套**：实验结果已登记在 `artifacts/shared/experiment_results.csv`（id 前缀 `memberB-`）。

## 结果一览

| 实验 | 内容 | SMAPE+1 |
|---|---|---|
| 参照 | 永远猜全局平均分 | 约 84 |
| **Baseline** | Ridge，只用 3 个元信息（预测哪项、出发月份、预测多远） | **71.58** |
| 改进 1 | LightGBM 用全部 1202 个特征，默认参数 | 71.23 |
| **改进 2** | LightGBM 调参（学习率 0.03、800 棵树、特征采样 0.5） | **70.75** |

**结论**：模型侧改进2比Baseline好了 0.83 分；seed为4486。

**数据预处理**：对数变换、独热编码、缺失值用训练折中位数填——细节见下面第 1 节。


In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '..')   # notebook 在 memberB 里，公共代码 common 在上一层，告诉 Python 去上一层找

from common.data import load_raw
from common.interface import model_folds

# 读入原始数据 + 组里冻结好的五折划分（samples.csv / patient_folds.csv）
raw = load_raw('../amp-parkinsons-disease-progression-prediction')
samples = pd.read_csv('../artifacts/shared/samples.csv')
patients = pd.read_csv('../artifacts/shared/patient_folds.csv')

# 把五折数据全部构建出来，之后所有实验都从这里取。第一次跑约 1 分钟。
folds = list(model_folds(raw, samples, patients))
print("五折都准备好了：", len(folds))


五折都准备好了： 5


## 1. 预处理

下面这个函数把每一折的训练/验证数据洗成模型能吃的样子，三个 Baseline 之后的实验都用它：
- 组学强度值先取对数（原始值跨 7 个数量级，不取对数模型会数值失稳）
- "预测哪项评分"做独热编码（变成 4 个 0/1 列；直接编 0/1/2/3 会让模型以为编号越大分越高，是错的）
- 空值用训练折的中位数填（中位数只能从训练折算，不能用验证集的信息）


In [2]:
KEY_COLS = ["visit_id", "patient_id", "visit_month", "horizon", "target", "target_month", "prediction_id"]

def prep_features(d):
    Xtr = d.train_features.drop(columns=KEY_COLS).copy()
    Xva = d.valid_features.drop(columns=KEY_COLS).copy()
    for df in (Xtr, Xva):
        for c in df.columns:
            if c.startswith(("protein::", "peptide::")):
                df[c] = np.log1p(df[c])                    # 修复①：强度值跨数量级 → 取对数
    Xtr = pd.concat([Xtr, pd.get_dummies(d.train_features["target"], prefix="t").astype(float)], axis=1)
    Xva = pd.concat([Xva, pd.get_dummies(d.valid_features["target"], prefix="t").astype(float)], axis=1)  # 修复②：独热编码
    med = Xtr.median()
    return Xtr.fillna(med), Xva.fillna(med)                # 修复③：缺失值=训练折中位数

## 2. Baseline：Ridge（只用元信息）

先用最简单的模型立个锚：只用"预测哪项评分 + 出发月份 + 预测多远"这 3 个信息，线性模型（Ridge）。
为什么不用组学特征：1195 列组学数据对线性模型基本是噪声，用了反而更差（实验记录里fold间波动很大）。
**结果：五折平均 71.58**，比参照（约 84）好 12 分。


In [3]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from common.metric import evaluate_oof
import time


t0 = time.time()
oof = []
for d in folds:
    Xtr, Xva = prep_features(d)
    meta = [c for c in Xtr.columns if c.startswith("t_updrs") or c in ("visit_month", "horizon")]
    sc = StandardScaler().fit(Xtr[meta])                   # 标准化（修复④，虽然这3列量级不大，养成习惯）
    m = Ridge(alpha=1.0).fit(sc.transform(Xtr[meta]), d.train_samples["rating"])
    pred = np.clip(m.predict(sc.transform(Xva[meta])), 0, None)   # 裁掉负数
    oof.append(pd.DataFrame({"prediction_id": d.valid_features["prediction_id"], "rating": pred}))
ridge_sec = round(time.time() - t0, 1)

eval_ridge = evaluate_oof(samples, pd.concat(oof))
print("Baseline Ridge 五折结果:", eval_ridge)

Baseline Ridge 五折结果: {'fold_scores': {0: 66.12651056694682, 1: 72.69165867936398, 2: 71.23513610394212, 3: 76.21275508755991, 4: 71.61830463299174}, 'mean_score': 71.57687301416091, 'std_score': 3.2421937381547967, 'oof_score': 71.41215420712409, 'n_predictions': 12697}


## 3. 改进 1：LightGBM 默认参数

**想法**：组学里可能藏着一点线性模型看不到的信号，树模型擅长抓"超过某个阈值才有影响"这种关系，试试看。
**结果：71.23**，只比 Baseline 好 0.35 分，而且各折忽好忽坏——说明信号很弱，默认参数也不稳。


In [4]:
import lightgbm as lgb, re

t0 = time.time()
oof = []
for d in folds:
    Xtr, Xva = prep_features(d)
    ren = lambda c: re.sub(r'[^0-9A-Za-z_]+', '_', c)   # LightGBM 不认列名里的 ::，先改名
    Xtr = Xtr.rename(columns=ren); Xva = Xva.rename(columns=ren)
    m = lgb.train(dict(objective="regression", verbosity=-1, seed=4486,
                       learning_rate=0.05, num_leaves=63, min_data_in_leaf=50),
                  lgb.Dataset(Xtr, label=d.train_samples["rating"]), num_boost_round=400)
    pred = np.clip(m.predict(Xva), 0, None)
    oof.append(pd.DataFrame({"prediction_id": d.valid_features["prediction_id"], "rating": pred}))
lgbm_sec = round(time.time() - t0, 1)

eval_lgbm = evaluate_oof(samples, pd.concat(oof))
print("LightGBM 五折结果:", eval_lgbm)

LightGBM 五折结果: {'fold_scores': {0: 63.65343522928484, 1: 73.26050832364997, 2: 71.6296626467955, 3: 74.37765273578039, 4: 73.23666881061405}, 'mean_score': 71.23158554922495, 'std_score': 3.8889960889469335, 'oof_score': 71.03580814697659, 'n_predictions': 12697}


## 4. 改进 2：LightGBM 调参

**改进**：学习率减半（0.05→0.03）、树加倍（400→800）、每棵树只看一半特征（防噪声列带偏）。
**结果：70.75，比 Baseline 好 0.83 分**，五折中 3 折改善。换 seed（42） 重跑时改进幅度在 0.8~1.6 之间波动——调参有稳定的小幅收益，但组学信号本身弱且对随机性敏感。


In [5]:
t0 = time.time()
oof = []
for d in folds:
    Xtr, Xva = prep_features(d)
    ren = lambda c: re.sub(r'[^0-9A-Za-z_]+', '_', c)
    Xtr = Xtr.rename(columns=ren); Xva = Xva.rename(columns=ren)
    m = lgb.train(dict(objective="regression", verbosity=-1, seed=4486,
                       learning_rate=0.03,          # 改①：小步慢学
                       num_leaves=63, min_data_in_leaf=50,
                       feature_fraction=0.5),       # 改②：每棵树只看随机一半特征，防噪声列带偏
                  lgb.Dataset(Xtr, label=d.train_samples["rating"]),
                  num_boost_round=800)              # 改③：树多近四倍，配低学习率
    pred = np.clip(m.predict(Xva), 0, None)
    oof.append(pd.DataFrame({"prediction_id": d.valid_features["prediction_id"], "rating": pred}))
lgbm2_sec = round(time.time() - t0, 1)

eval_lgbm2 = evaluate_oof(samples, pd.concat(oof))
print("LightGBM 调参版 五折结果:", eval_lgbm2)

LightGBM 调参版 五折结果: {'fold_scores': {0: 62.75550722749613, 1: 73.58246545234938, 2: 70.12791401447292, 3: 74.43169113200166, 4: 72.84377900875648}, 'mean_score': 70.74827136701532, 'std_score': 4.248776562491079, 'oof_score': 70.53557094466491, 'n_predictions': 12697}


## 5. 登记实验日志

把三条实验追加进三人共用的日志。experiment_id 不能重复，重跑要换新名字。


In [6]:
import json
from common.experiments import log_experiment

split_id = json.load(open("../artifacts/shared/manifest.json"))["split_id"]
NB = "detail in memberB/view.ipynb"

log_experiment("../artifacts/shared/experiment_results.csv",
    experiment_id="memberB-ridge-meta-v1", model="Ridge(alpha=1.0)",
    features="one-hot(target) + visit_month + horizon (NO omics); median-imputed from train fold",
    evaluation=eval_ridge, seed=4486, runtime_seconds=ridge_sec, split_id=split_id,
    config={"alpha": 1.0},
    notes=f"Baseline, deterministic. {NB}")

log_experiment("../artifacts/shared/experiment_results.csv",
    experiment_id="memberB-lightgbm-default-v1", model="LightGBM(default)",
    features="all 1202: omics log1p + one-hot(target) + visit_month/horizon, train-fold median impute",
    evaluation=eval_lgbm, seed=4486, runtime_seconds=lgbm_sec, split_id=split_id,
    config={"lr": 0.05, "num_leaves": 63, "rounds": 400},
    notes=f"Only marginal gain, unstable across folds. {NB}")

log_experiment("../artifacts/shared/experiment_results.csv",
    experiment_id="memberB-lightgbm-tuned-v1", model="LightGBM(tuned)",
    features="all 1202: omics log1p + one-hot(target) + visit_month/horizon, train-fold median impute",
    evaluation=eval_lgbm2, seed=4486, runtime_seconds=lgbm2_sec, split_id=split_id,
    config={"lr": 0.03, "num_leaves": 63, "rounds": 800, "feature_fraction": 0.5},
    notes=f"Model-side improvement: tuned params, see eval for delta vs baseline. {NB}")